# GPT Vision API 테스트 노트북
> FirstCare ML2: 식단 사진 분석 + 운동 캡처 인증

## 구조
1. 환경 설정 & API 키 로드
2. 유틸리티 함수
3. API 연결 테스트
4. 식단 사진 분석 — 무료 (REQ-HLTH-007)
5. 식단 사진 분석 — 유료 상세 리포트 (REQ-HLTH-007, -300pt)
6. 운동 캡처 OCR (REQ-CHAL-009)


## 1. 환경 설정

In [ ]:
# # 필요한 패키지 설치 (최초 1회)
# !pip install openai python-dotenv

In [4]:
import os
import base64
import json
from openai import OpenAI
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("API 키 로드 완료!")

API 키 로드 완료!


## 2. 유틸리티 함수

In [5]:
def encode_image(image_path: str) -> str:
    """이미지 파일을 base64로 변환"""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def get_image_media_type(image_path: str) -> str:
    """파일 확장자로 미디어 타입 결정"""
    ext = image_path.lower().split(".")[-1]
    media_types = {
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "png": "image/png",
        "gif": "image/gif",
        "webp": "image/webp"
    }
    return media_types.get(ext, "image/jpeg")

def call_vision_api(image_path: str, system_prompt: str, user_text: str, model: str = "gpt-4o-mini", detail: str = "low"):
    """Vision API 호출 공통 함수"""
    base64_image = encode_image(image_path)
    media_type = get_image_media_type(image_path)
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{base64_image}",
                            "detail": detail
                        }
                    },
                    {
                        "type": "text",
                        "text": user_text
                    }
                ]
            }
        ],
        max_tokens=1000
    )
    return response

def parse_and_print(response):
    """응답 파싱 및 출력"""
    raw = response.choices[0].message.content
    print("=== Raw 응답 ===")
    print(raw)
    print()
    
    # JSON 파싱
    try:
        result = json.loads(raw)
        print("=== 파싱 결과 ===")
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return result
    except json.JSONDecodeError:
        print("⚠️ JSON 파싱 실패 - prompt 수정 필요")
        return None

def print_usage(response, model="gpt-4o-mini"):
    """토큰 사용량 및 비용 출력"""
    usage = response.usage
    print(f"Input tokens:  {usage.prompt_tokens}")
    print(f"Output tokens: {usage.completion_tokens}")
    print(f"Total tokens:  {usage.total_tokens}")
    
    if model == "gpt-4o-mini":
        input_cost = usage.prompt_tokens / 1_000_000 * 0.15
        output_cost = usage.completion_tokens / 1_000_000 * 0.60
    else:  # gpt-4o
        input_cost = usage.prompt_tokens / 1_000_000 * 2.50
        output_cost = usage.completion_tokens / 1_000_000 * 10.00
    
    print(f"예상 비용: ${input_cost + output_cost:.6f}")

print("유틸리티 함수 로드 완료!")

유틸리티 함수 로드 완료!


## 3. API 연결 테스트

In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "테스트입니다. '연결 성공'이라고만 답해주세요."}
    ],
    max_tokens=50
)

print(response.choices[0].message.content)

연결 성공


## 4. 식단 사진 분석 — 무료 (REQ-HLTH-007)

> 음식 사진 → 탄/단/지 비율 % + 한 줄 피드백
> 
> 모델: GPT-4o-mini | detail: low | 비용: ~$0.0002/장


In [7]:
# 식단 분석 — 무료 prompt
MEAL_FREE_PROMPT = """당신은 영양 상담사입니다.
사용자가 보낸 한 끼 식사 사진을 보고 영양 피드백을 제공합니다.
 
아래 JSON 형식으로만 응답하세요. JSON 외의 텍스트는 절대 포함하지 마세요.
 
{
    "food_name": "대표 음식 이름",
    "food_items": ["개별 음식1", "개별 음식2"],
    "nutrition_ratio": {
        "carbohydrate_pct": 0,
        "protein_pct": 0,
        "fat_pct": 0
    },
    "sodium_level": "높음/보통/낮음",
    "feedback": "피드백 (2문장: 팩트 1문장 + 제안 1문장)"
}
 
피드백 작성 원칙:
- 첫 문장: 사실 기반 평가 ("~입니다", "~한 편입니다")
- 두 번째 문장: 개선 제안 ("~해보세요", "~을 곁들이면 좋습니다")
- "위험", "주의", "과다" 같은 경고 표현 금지
- "좋은 선택이에요", "훌륭해요" 같은 과한 칭찬 금지
- 한 끼 식사 기준으로 평가
- 예시: "단백질이 충분합니다. 다만 포화지방 비율이 높은 편이니, 다음 식사에 채소를 늘려보세요."
 
음식 인식 기준:
- 모든 종류의 음식을 인식할 것 (한식, 양식, 중식, 일식, 디저트, 간식 등)
- 한국 음식을 정확히 구분할 것:
  · 철판/불판 위 둥근 모양 내장류 → 곱창, 대창, 막창으로 인식
  · 얇게 썬 구운 고기 → 삼겹살, 목살, 갈비 등 부위 구분
  · 국/찌개/탕 종류 정확히 구분 (김치찌개, 된장찌개, 순두부 등)
  · 분식류 구분 (떡볶이, 순대, 튀김, 김밥 등)
- 여러 음식이 함께 있으면 food_items에 각각 나열
 
분석 기준:
- nutrition_ratio 세 값의 합은 100
- sodium_level 판단:
  · 높음: 국/찌개, 라면, 젓갈, 장류, 내장구이(양념)
  · 보통: 일반 반찬, 구이류
  · 낮음: 샐러드, 과일, 나물 위주
- 음식 사진이 아닌 경우: food_name을 "인식 불가"로 설정
- 한국어로 응답
"""


print("무료 식단 분석 prompt 설정 완료!")

무료 식단 분석 prompt 설정 완료!


In [8]:
# 식단 사진 분석 실행 — 무료
# ⬇️ 테스트할 음식 사진 경로
IMAGE_PATH = "test_images/튀김.jpg"

response = call_vision_api(
    image_path=IMAGE_PATH,
    system_prompt=MEAL_FREE_PROMPT,
    user_text="이 음식 사진을 분석해주세요.",
    model="gpt-4o-mini",
    detail="high"
)

result = parse_and_print(response)
print()
print_usage(response, model="gpt-4o-mini")

=== Raw 응답 ===
{
    "food_name": "철판 소시지와 감자튀김",
    "food_items": ["소시지", "감자튀김", "샐러드"],
    "nutrition_ratio": {
        "carbohydrate_pct": 40,
        "protein_pct": 30,
        "fat_pct": 30
    },
    "sodium_level": "높음",
    "feedback": "단백질과 탄수화물이 적절히 포함되어 있습니다. 그러나 나트륨이 높으니, 다음 식사에는 저염식 재료를 활용해보세요."
}

=== 파싱 결과 ===
{
  "food_name": "철판 소시지와 감자튀김",
  "food_items": [
    "소시지",
    "감자튀김",
    "샐러드"
  ],
  "nutrition_ratio": {
    "carbohydrate_pct": 40,
    "protein_pct": 30,
    "fat_pct": 30
  },
  "sodium_level": "높음",
  "feedback": "단백질과 탄수화물이 적절히 포함되어 있습니다. 그러나 나트륨이 높으니, 다음 식사에는 저염식 재료를 활용해보세요."
}

Input tokens:  9101
Output tokens: 130
Total tokens:  9231
예상 비용: $0.001443


## 5. 식단 사진 분석 — 유료 상세 리포트 (REQ-HLTH-007, -300pt)

> 무료 분석 결과를 기반으로 상세 리포트 생성
> 
> 부족한 영양소를 채울 수 있는 음식 추천 포함


In [9]:
# 식단 분석 — 유료 상세 리포트 prompt
MEAL_PAID_PROMPT = """당신은 영양 상담사입니다.
사용자가 보낸 한 끼 식사 사진을 상세 분석하여 실천 가능한 개선 방안을 제공합니다.

아래 JSON 형식으로만 응답하세요. JSON 외의 텍스트는 절대 포함하지 마세요.

{
    "food_name": "대표 음식 이름",
    "food_items": ["개별 음식1", "개별 음식2"],
    "nutrition_ratio": {
        "carbohydrate_pct": 0,
        "protein_pct": 0,
        "fat_pct": 0
    },
    "sodium_level": "높음/보통/낮음",
    "detailed_analysis": {
        "strength": "이 식단의 장점 (1문장)",
        "improvement": "개선할 점 + 제안 (1~2문장)"
    },
    "recommendations": [
        {
            "nutrient": "보충하면 좋은 영양소",
            "foods": ["추천 음식1", "추천 음식2", "추천 음식3"],
            "reason": "추천 이유 (1문장)"
        }
    ],
    "next_meal_suggestion": {
        "concept": "다음 끼니 컨셉 (예: 채소 중심 저염식)",
        "menu_example": ["추천 메뉴1", "추천 메뉴2"],
        "reason": "이유 (1문장)"
    },
    "overall_score": 0,
    "feedback_summary": "종합 피드백 (2문장: 팩트 요약 + 실천 제안)"
}

피드백 작성 원칙:
- 탄/단/지 비율이 균형 잡힌 경우 (탄 40~60, 단 20~35, 지 15~30):
  사실 기반으로 인정하되 담백하게.  예: "탄수화물, 단백질, 지방의 비율이 균형 잡힌 식사입니다. 이 패턴을 유지해보세요."   
- 그 외: 기존대로 팩트 1문장 + 제안 1문장

- 좋은 점은 간단히 인정 (과하게 칭찬하지 말 것 -> 적당히 칭찬할 것)
- 개선점은 솔직하게 짚되, "~입니다" 팩트 전달 후 "~해보세요" 제안으로 마무리
- "위험", "주의", "과다" 같은 경고 표현 금지
- "좋은 선택이에요", "훌륭해요" 같은 과한 칭찬 금지


- 한 끼 식사 기준으로 평가

음식 인식 기준:
- 모든 종류의 음식을 인식할 것 (한식, 양식, 중식, 일식, 디저트, 간식 등)
- 한국 음식을 정확히 구분할 것:
  · 철판/불판 위 둥근 모양 내장류 → 곱창, 대창, 막창
  · 얇게 썬 구운 고기 → 삼겹살, 목살, 갈비 등 부위 구분
  · 국/찌개/탕, 분식류 정확히 구분
- 여러 음식이 함께 있으면 food_items에 각각 나열

점수 기준 (overall_score, 1~10):
- - 8점 이상: improvement에 개선점 대신 유지 팁 제공
  예: "균형 잡힌 구성입니다. 이 식단 패턴을 유지해보세요."
  recommendations는 "보충" 대신 "이 식단과 잘 어울리는 음식" 추천
- 5~7: 괜찮지만 보완 여지 있음
- 3~4: 한쪽으로 치우친 식사
- 1~2: 영양 균형이 많이 부족

분석 기준:
- nutrition_ratio 세 값의 합은 100
- recommendations는 2개 작성
- next_meal_suggestion 메뉴는 한국에서 쉽게 먹을 수 있는 것
- 한국어로 응답
"""

print("유료 식단 분석 prompt 설정 완료!")

유료 식단 분석 prompt 설정 완료!


In [10]:
# 식단 사진 분석 실행 — 유료 상세 리포트
# ⬇️ 테스트할 음식 사진 경로 (무료와 같은 이미지로 비교)
IMAGE_PATH = "test_images/연어포케.jpg"

response = call_vision_api(
    image_path=IMAGE_PATH,
    system_prompt=MEAL_PAID_PROMPT,
    user_text="이 음식 사진을 상세 분석해주세요.",
    model="gpt-4o-mini",
    detail="high"
)

result = parse_and_print(response)
print()
print_usage(response, model="gpt-4o-mini")

=== Raw 응답 ===
{
    "food_name": "연어 샐러드 볼",
    "food_items": ["연어", "아보카도", "삶은 계란", "상추", "체리 토마토", "검은 올리브", "병아리콩", "양파", "샐러드 채소"],
    "nutrition_ratio": {
        "carbohydrate_pct": 30,
        "protein_pct": 35,
        "fat_pct": 35
    },
    "sodium_level": "보통",
    "detailed_analysis": {
        "strength": "야채와 단백질이 풍부한 균형 잡힌 식사입니다.",
        "improvement": "탄수화물 비율이 상대적으로 낮습니다. 고구마나 퀴노아를 추가해보세요."
    },
    "recommendations": [
        {
            "nutrient": "식이섬유와 탄수화물",
            "foods": ["고구마", "퀴노아", "현미"],
            "reason": "식사에 섬유소와 탄수화물을 보충하여 영양 균형을 맞출 수 있습니다."
        },
        {
            "nutrient": "비타민 C",
            "foods": ["파프리카", "브로콜리", "귤"],
            "reason": "비타민 C를 추가하면 면역력 강화와 건강 유지에 도움이 됩니다."
        }
    ],
    "next_meal_suggestion": {
        "concept": "곡물 기반 균형식",
        "menu_example": ["유부초밥", "야채잡채"],
        "reason": "곡물과 다양한 영양소를 포함한 식사를 통해 균형 잡힌 영양을 유지할 수 있습니다."
    },
    "overall_score": 8,
    "feedback_summary

## 6. 운동 캡처 OCR (REQ-CHAL-009)

> 운동 앱 스크린샷 → 운동 데이터 추출 + 챌린지 자동 완료
> 
> 모델: GPT-4o | detail: high | 비용: ~$0.0019/장


In [11]:
# 운동 캡처 인증 prompt
EXERCISE_PROMPT = """당신은 운동 앱 스크린샷을 분석하는 전문 AI입니다.
 
한국에서 주로 사용하는 운동 앱의 UI를 정확히 인식합니다:
- 삼성헬스 (Samsung Health): 걸음수, 달리기, 사이클링 등
- 나이키런클럽 (Nike Run Club): 러닝 기록
- 애플헬스 (Apple Health): 종합 건강 데이터
- 카카오헬스케어 (Kakao Healthcare)
- 스트라바 (Strava): 러닝, 사이클링
 
사용자가 스크린샷을 보내면, 아래 JSON 형식으로만 응답하세요.
JSON 외의 텍스트는 절대 포함하지 마세요.
 
{
    "app_name": "앱 이름",
    "exercise_type": "운동 종류 (걷기/달리기/자전거/수영/등산/기타)",
    "metrics": {
        "steps": null,
        "distance_km": null,
        "duration_minutes": null,
        "calories_burned": null,
        "pace_per_km": null
    },
    "date": "기록 날짜 (읽을 수 있는 경우, YYYY-MM-DD)",
    "is_verified": true,
    "confidence": "high/medium/low",
    "note": "특이사항"
}
 
분석 규칙:
- 숫자를 최대한 정확하게 읽을 것
  · 쉼표 구분 숫자 주의: 11,166 → 11166
  · 시간 형식 변환: 32:02 → 32분, 1:05:30 → 65.5분
  · 거리 단위 확인: km인지 m인지 구분
  · 페이스: 07'03"/km 같은 형식도 인식
- pace_per_km: "분'초\"" 형식으로 표시 (예: "7'03\"")
- 한국어 UI 텍스트 인식:
  · "걸음" = steps
  · "분" = minutes
  · "칼로리" / "kcal" = calories
  · "킬로미터" / "km" = distance
- is_verified 판단:
  · true: 운동 앱 스크린샷이 확실한 경우
  · false: 운동 앱이 아니거나, 조작이 의심되는 경우
- confidence 판단:
  · high: 모든 수치를 명확히 읽을 수 있음
  · medium: 일부 수치가 불확실함
  · low: 대부분 수치를 읽기 어려움
- 운동 앱이 아닌 이미지: is_verified를 false, note에 "운동 앱 스크린샷이 아닙니다" 기재
- 한국어로 응답
"""

print("운동 캡처 인증 prompt 설정 완료!")

운동 캡처 인증 prompt 설정 완료!


In [15]:
# 운동 캡처 분석 실행
# ⬇️ 테스트할 운동 앱 스크린샷 경로
IMAGE_PATH = "test_images/exercise_삼성헬스.png"

response = call_vision_api(
    image_path=IMAGE_PATH,
    system_prompt=EXERCISE_PROMPT,
    user_text="이 운동 앱 스크린샷을 분석해주세요.",
    model="gpt-4o-mini",
    detail="high"
)

result = parse_and_print(response)
print()
print_usage(response, model="gpt-4o-mini")

=== Raw 응답 ===
{
    "app_name": "삼성헬스",
    "exercise_type": "달리기",
    "metrics": {
        "steps": 11166,
        "distance_km": 4.53,
        "duration_minutes": 32,
        "calories_burned": 365,
        "pace_per_km": "7'03\""
    },
    "date": null,
    "is_verified": true,
    "confidence": "high",
    "note": ""
}

=== 파싱 결과 ===
{
  "app_name": "삼성헬스",
  "exercise_type": "달리기",
  "metrics": {
    "steps": 11166,
    "distance_km": 4.53,
    "duration_minutes": 32,
    "calories_burned": 365,
    "pace_per_km": "7'03\""
  },
  "date": null,
  "is_verified": true,
  "confidence": "high",
  "note": ""
}

Input tokens:  14777
Output tokens: 107
Total tokens:  14884
예상 비용: $0.002281
